<a href="https://colab.research.google.com/github/rm571222/dataholics-oracle-challenge/blob/main/notebooks/02_data_upload/nb5_upload_internacoes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB5 — Carga no Oracle: Internações Hospitalares (T_SIH_INTERNACAO)

**Projeto DATAHOLICS — FIAP Challenge | Parceria Oracle**

Este notebook documenta a carga da tabela fato de internações no Oracle Autonomous AI Database, com base nas decisões tomadas na fase de exploração (NB1). Cobre: criação da estrutura já com nomenclatura padronizada e chave primária composta, extração e carga do período completo, investigação de uma lacuna na fonte, e validação final 1:1 entre a fonte e o banco.

## Estrutura deste notebook
1. Bibliotecas e autenticação
2. Modelagem e criação da tabela (DDL)
3. Extração e carga do período completo
4. Identificação e solução de uma lacuna na fonte
5. Validação final 1:1 (fonte vs. banco)

## 1. Bibliotecas e autenticação

Conexão via wallet do Autonomous Database. Desativamos o paralelismo da sessão para evitar deadlocks em operações de carga em lote — uma característica observada do Autonomous Database ao processar grandes volumes concorrentemente.

In [ ]:
!pip install -q oracledb pysus pyreaddbc dbfread

from google.colab import files
uploaded = files.upload()  # selecione o Wallet_SPRINT02CHALLENGE.zip

import zipfile, os
wallet_path = '/content/wallet'
os.makedirs(wallet_path, exist_ok=True)
with zipfile.ZipFile(list(uploaded.keys())[0], 'r') as z:
    z.extractall(wallet_path)
print(os.listdir(wallet_path))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.0/334.0 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.5/316.5 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 M

Saving Wallet_SPRINT02CHALLENGE.zip to Wallet_SPRINT02CHALLENGE.zip
['tnsnames.ora', 'keystore.jks', 'README', 'ojdbc.properties', 'cwallet.sso', 'truststore.jks', 'ewallet.pem', 'ewallet.p12', 'sqlnet.ora']


In [ ]:
import oracledb, pandas as pd, math
from pysus import sih
from ftplib import FTP
import pyreaddbc
from dbfread import DBF
import getpass

senha_banco = getpass.getpass("Senha do banco: ")
senha_wallet = getpass.getpass("Senha do wallet: ")

conn = oracledb.connect(
    user="ADMIN",
    password=senha_banco,
    dsn="sprint02challenge_high",
    config_dir=wallet_path,
    wallet_location=wallet_path,
    wallet_password=senha_wallet
)

cursor = conn.cursor()
cursor.execute("ALTER SESSION DISABLE PARALLEL DML")
cursor.execute("ALTER SESSION DISABLE PARALLEL QUERY")
print("Conectado com sucesso!")

PySUS 2.10.6 -- welcome!
Data cache: /root/pysus
Change it with: pysus.set_cache('/your/path')
Browse datasets with: pysus.info()

Senha do banco: ··········
Senha do wallet: ··········
Conectado com sucesso!


In [ ]:
def inserir_em_lotes(cursor, conn, sql, dados, tamanho_lote=20000):
    """Insere em blocos, usando batcherrors para não interromper a carga
    caso alguma linha pontual viole a chave primária."""
    total_ignorados = 0
    for i in range(0, len(dados), tamanho_lote):
        lote = dados[i:i + tamanho_lote]
        cursor.executemany(sql, lote, batcherrors=True)
        erros = cursor.getbatcherrors()
        if erros:
            total_ignorados += len(erros)
        conn.commit()
    if total_ignorados > 0:
        print(f'  ({total_ignorados} linhas ignoradas por chave primária duplicada)')
    return total_ignorados

def baixar_e_converter_dbc(ano_mes, pasta_destino='/content', timeout=60):
    """Baixa um arquivo RD (.dbc) direto do FTP oficial do DATASUS e converte
    para DataFrame — usado quando o catálogo do pysus ainda não processou
    a competência, mas o arquivo bruto já existe na fonte primária."""
    nome_arquivo = f'RDSP{ano_mes}.dbc'
    caminho_dbc = f'{pasta_destino}/{nome_arquivo}'
    caminho_dbf = caminho_dbc.replace('.dbc', '.dbf')
    ftp = FTP('ftp.datasus.gov.br', timeout=timeout)
    ftp.login()
    ftp.cwd('/dissemin/publicos/SIHSUS/200801_/Dados/')
    with open(caminho_dbc, 'wb') as f:
        ftp.retrbinary(f'RETR {nome_arquivo}', f.write)
    ftp.quit()
    pyreaddbc.dbc2dbf(caminho_dbc, caminho_dbf)
    tabela = DBF(caminho_dbf, encoding='latin1')
    return pd.DataFrame(iter(tabela))

## 2. Modelagem e criação da tabela (DDL)

A tabela nasce diretamente com a nomenclatura padronizada e com **chave primária composta** (`nr_aih` + competência). Essa decisão de PK vem de uma característica real do SIH/SUS: uma mesma AIH de longa permanência (`cd_tipo_aih = 5`) gera um novo registro a cada mês em que o paciente permanece internado, mantendo o mesmo número de AIH — por isso `nr_aih` sozinho não garante unicidade (BEZERRA, 2020).

As 21 colunas selecionadas na fase de exploração (NB1) cobrem identificação, perfil do paciente, diagnóstico, datas/desfecho, capacidade (UTI/permanência) e financeiro — o suficiente para responder às perguntas de negócio do projeto sem carregar as 93 colunas restantes do layout original.

In [ ]:
cursor.execute("""
    BEGIN
        EXECUTE IMMEDIATE 'DROP TABLE T_SIH_INTERNACAO';
    EXCEPTION WHEN OTHERS THEN IF SQLCODE != -942 THEN RAISE; END IF;
    END;
""")

cursor.execute("""
    CREATE TABLE T_SIH_INTERNACAO (
        nr_aih                      VARCHAR2(20),
        cd_hospital                 VARCHAR2(10),
        cd_municipio_hospital       VARCHAR2(6),
        cd_municipio_residencia     VARCHAR2(6),
        nr_ano_competencia          NUMBER(4),
        nr_mes_competencia          NUMBER(2),
        cd_tipo_aih                 NUMBER(1),
        sg_sexo                     VARCHAR2(1),
        nr_idade                    NUMBER(3),
        cd_raca_cor                 VARCHAR2(2),
        cd_diagnostico_principal    VARCHAR2(10),
        cd_especialidade_leito      VARCHAR2(2),
        dt_internacao               DATE,
        dt_saida                    DATE,
        qt_dias_permanencia         NUMBER(5),
        fl_obito                    NUMBER(1),
        cd_carater_internacao       VARCHAR2(2),
        qt_dias_uti                 NUMBER(3),
        vl_uti                      NUMBER(12,2),
        vl_total_internacao         NUMBER(12,2),
        cd_complexidade             VARCHAR2(2),
        CONSTRAINT PK_SIH_INTERNACAO PRIMARY KEY (nr_aih, nr_ano_competencia, nr_mes_competencia)
    )
""")
conn.commit()
print("Tabela T_SIH_INTERNACAO criada")

## 3. Extração e carga do período completo

Período: junho/2024 a junho/2026 (25 competências). Antes da carga, o `SEXO` é recodificado para `'0'` (Não informado) quando fora do domínio válido — decisão tomada na exploração (NB1), preservando o registro da internação em vez de descartá-lo.

In [ ]:
COLUNAS_ORIGEM = ['N_AIH','CNES','MUNIC_MOV','MUNIC_RES','ANO_CMPT','MES_CMPT','IDENT',
                   'SEXO','IDADE','RACA_COR','DIAG_PRINC','ESPEC','DT_INTER','DT_SAIDA',
                   'DIAS_PERM','MORTE','CAR_INT','UTI_MES_TO','VAL_UTI','VAL_TOT','COMPLEX']

COLUNAS_BANCO = ['nr_aih','cd_hospital','cd_municipio_hospital','cd_municipio_residencia',
                  'nr_ano_competencia','nr_mes_competencia','cd_tipo_aih','sg_sexo','nr_idade',
                  'cd_raca_cor','cd_diagnostico_principal','cd_especialidade_leito',
                  'dt_internacao','dt_saida','qt_dias_permanencia','fl_obito',
                  'cd_carater_internacao','qt_dias_uti','vl_uti','vl_total_internacao','cd_complexidade']

INSERT_SQL = f"""
    INSERT INTO T_SIH_INTERNACAO ({','.join(COLUNAS_BANCO)})
    VALUES ({','.join([':'+str(i+1) for i in range(len(COLUNAS_BANCO))])})
"""

PERIODO = [(2024, m) for m in range(6, 13)] + [(2025, m) for m in range(1, 13)] + [(2026, m) for m in range(1, 7)]

MESES_VIA_FTP = ['2505', '2507', '2510', '2603', '2605', '2606']

total_geral = 0
for ano, mes in PERIODO:
    ano_mes = f'{str(ano)[2:]}{mes:02d}'

    cursor.execute(
        "SELECT COUNT(*) FROM T_SIH_INTERNACAO WHERE nr_ano_competencia = :1 AND nr_mes_competencia = :2",
        [ano, mes]
    )
    if cursor.fetchone()[0] > 0:
        print(f'{ano}-{mes:02d}: já carregado, pulando')
        continue

    try:
        if ano_mes in MESES_VIA_FTP:
            df_mes = baixar_e_converter_dbc(ano_mes)
        else:
            caminhos = sih(state="SP", year=ano, month=mes)
            caminhos_rd = [p for p in caminhos if "RDSP" in p.upper()]
            if not caminhos_rd:
                print(f'{ano}-{mes:02d}: sem RD disponível')
                continue
            df_mes = pd.concat([pd.read_parquet(p) for p in caminhos_rd], ignore_index=True)

        for col in COLUNAS_ORIGEM:
            if col not in df_mes.columns:
                df_mes[col] = None
        df_mes = df_mes[COLUNAS_ORIGEM].copy()

        df_mes['SEXO'] = df_mes['SEXO'].astype(str).str.strip().where(
            df_mes['SEXO'].astype(str).str.strip().isin(['0', '1', '3']), '0'
        )

        for col in ['DT_INTER', 'DT_SAIDA']:
            df_mes[col] = pd.to_datetime(df_mes[col], format='%Y%m%d', errors='coerce')
        for col in ['IDADE','DIAS_PERM','MORTE','UTI_MES_TO','VAL_UTI','VAL_TOT','ANO_CMPT','MES_CMPT','IDENT']:
            df_mes[col] = pd.to_numeric(df_mes[col], errors='coerce')

        df_mes = df_mes.where(pd.notnull(df_mes), None)

        dados = [tuple(x.item() if hasattr(x, 'item') else x for x in row)
                 for row in df_mes.itertuples(index=False, name=None)]
        inserir_em_lotes(cursor, conn, INSERT_SQL, dados)

        total_geral += len(dados)
        print(f'{ano}-{mes:02d}: {len(dados)} linhas carregadas (total: {total_geral})')

    except Exception as e:
        print(f'{ano}-{mes:02d} falhou: {e}')
        conn.rollback()

cursor.execute("SELECT COUNT(*) FROM T_SIH_INTERNACAO")
print(f"\nTotal final na tabela: {cursor.fetchone()[0]}")

## 4. Identificação e solução de uma lacuna na fonte

Seis competências não tinham arquivo RD disponível no catálogo do `pysus` no momento da extração: maio, julho e outubro de 2025, e março, maio e junho de 2026. Como o padrão não seguia uma sequência lógica de atraso, investigamos antes de aceitar como definitivo.

In [ ]:
arquivos_verificar = ['RDSP2505.dbc', 'RDSP2507.dbc', 'RDSP2510.dbc',
                       'RDSP2603.dbc', 'RDSP2605.dbc', 'RDSP2606.dbc']

ftp = FTP('ftp.datasus.gov.br', timeout=15)
ftp.login()
ftp.cwd('/dissemin/publicos/SIHSUS/200801_/Dados/')
arquivos = ftp.nlst()

for nome_esperado in arquivos_verificar:
    existe = any(nome_esperado.lower() in a.lower() for a in arquivos)
    print(f'{nome_esperado}: {"existe no FTP oficial" if existe else "NÃO existe"}')

ftp.quit()

**Resultado:** todos os 6 arquivos estavam publicados no FTP oficial do DATASUS. O dado sempre existiu na fonte primária — a ausência era um atraso no catálogo interno do `pysus` (que converte os arquivos `.dbc` originais para parquet), não uma lacuna real do Ministério da Saúde. Por isso, esses 6 meses foram recuperados diretamente via FTP + `pyreaddbc` (mesma biblioteca de baixo nível que o próprio `pysus` usa internamente), já integrado ao loop da Seção 3.

## 5. Validação final 1:1 (fonte vs. banco)

Comparamos, mês a mês, três métricas independentes entre a fonte e o banco: quantidade de registros, valor financeiro total e óbitos. Uma pequena divergência residual foi investigada e confirmada como **duplicidade legítima de `N_AIH` dentro do próprio arquivo mensal do DATASUS** — não um problema de carga.

In [ ]:
def obter_metricas(df):
    linhas_extras = df[df['N_AIH'].duplicated(keep='first')]
    return {
        'qtd': len(df),
        'val_tot_soma': round(pd.to_numeric(df['VAL_TOT'], errors='coerce').sum(), 2),
        'morte_soma': int(pd.to_numeric(df['MORTE'], errors='coerce').sum()),
        'qtd_duplicatas': len(linhas_extras),
        'val_tot_duplicatas': round(pd.to_numeric(linhas_extras['VAL_TOT'], errors='coerce').sum(), 2),
        'morte_duplicatas': int(pd.to_numeric(linhas_extras['MORTE'], errors='coerce').sum())
    }

resultados_fonte = []
for ano, mes in PERIODO:
    ano_mes = f'{str(ano)[2:]}{mes:02d}'
    if ano_mes in MESES_VIA_FTP:
        df_mes = baixar_e_converter_dbc(ano_mes)
    else:
        caminhos = sih(state="SP", year=ano, month=mes)
        caminhos_rd = [p for p in caminhos if "RDSP" in p.upper()]
        if not caminhos_rd:
            continue
        df_mes = pd.concat([pd.read_parquet(p) for p in caminhos_rd], ignore_index=True)

    m = obter_metricas(df_mes)
    m['ano'], m['mes'] = ano, mes
    resultados_fonte.append(m)

df_fonte = pd.DataFrame(resultados_fonte)

In [ ]:
cursor.execute("""
    SELECT nr_ano_competencia AS ano, nr_mes_competencia AS mes,
           COUNT(*) AS qtd,
           ROUND(SUM(vl_total_internacao), 2) AS val_tot_soma,
           SUM(fl_obito) AS morte_soma
    FROM T_SIH_INTERNACAO
    GROUP BY nr_ano_competencia, nr_mes_competencia
    ORDER BY 1, 2
""")
colunas_result = [c[0].lower() for c in cursor.description]
df_oracle = pd.DataFrame(cursor.fetchall(), columns=colunas_result)
df_oracle = df_oracle.rename(columns={'qtd': 'qtd_oracle', 'val_tot_soma': 'val_tot_soma_oracle', 'morte_soma': 'morte_soma_oracle'})

In [ ]:
comparacao = df_fonte.merge(df_oracle, on=['ano', 'mes'])

comparacao['diff_qtd'] = comparacao['qtd'] - comparacao['qtd_oracle']
comparacao['diff_val_tot'] = comparacao['val_tot_soma'] - comparacao['val_tot_soma_oracle']
comparacao['diff_morte'] = comparacao['morte_soma'] - comparacao['morte_soma_oracle']

comparacao['qtd_bate'] = comparacao['diff_qtd'] == comparacao['qtd_duplicatas']
comparacao['val_bate'] = (comparacao['diff_val_tot'] - comparacao['val_tot_duplicatas']).abs() < 0.01
comparacao['morte_bate'] = comparacao['diff_morte'] == comparacao['morte_duplicatas']

comparacao['status'] = comparacao.apply(
    lambda r: 'OK' if (r['qtd_bate'] and r['val_bate'] and r['morte_bate']) else 'DIVERGENTE',
    axis=1
)

print(comparacao[['ano','mes','qtd','qtd_oracle','diff_qtd','qtd_duplicatas','status']])
print(f"\nMeses OK: {(comparacao['status'] == 'OK').sum()} de {len(comparacao)}")

**Resultado: 25 de 25 meses OK.** A divergência residual (0,036% do total) corresponde, centavo a centavo e óbito a óbito, à duplicidade legítima de `N_AIH` dentro do próprio arquivo mensal — fenômeno documentado na literatura acadêmica sobre o SIH/SUS (BEZERRA, 2020). A chave primária composta preserva corretamente essas linhas quando ocorrem entre competências diferentes (longa permanência), e rejeita apenas a segunda ocorrência quando a duplicidade acontece dentro da mesma competência — comportamento correto, não perda de dado.

> **Referência:** BEZERRA, Sarah Lima. *Detecção de Outliers na Produção do SIH/SUS sob a perspectiva dos atendimentos à população dos municípios brasileiros*. Trabalho de Conclusão de Curso — Instituto Serzedello Corrêa, Tribunal de Contas da União, 2020.

https://sites.tcu.gov.br/recursos/trabalhos-pos-graduacao/pdfs/Detec%C3%A7%C3%A3o%20de%20Outliers%20na%20produ%C3%A7%C3%A3o%20do%20SIH_SUS%20sob%20a%20perspectiva%20dos%20atendimentos%20%C3%A0%20popula%C3%A7%C3%A3o%20dos%20m.pdf

## Conclusão — NB5

A tabela `T_SIH_INTERNACAO` foi carregada com sucesso: 25 competências (jun/2024 a jun/2026), 6.107.861 registros, chave primária composta preservando corretamente internações de longa permanência, e validação 1:1 confirmada com 100% de aderência explicada. As foreign keys para as tabelas de domínio, CID-10 e hospital são criadas no NB7, após a carga dessas tabelas de apoio.